# 09 – Model data: choose the features, create dummy variables, mark the train / test periods
**Input:** `food_final.csv` (from `06_Merge.ipynb`) and `location_features.csv` (from `08_Location_Check.ipynb`)  **Output:** `model_data.csv`

This notebook prepares the data for the models in Phase 2. We keep only the columns that are **known before the inspection** (no leakage), turn the categorical columns into dummy variables, and keep the columns we need to split the data into training and test sets (Period, Placebo_Period).

In [ ]:
import pandas as pd
import numpy as np

pandas for the tables and numpy for numbers.

In [ ]:
final = pd.read_csv('food_final.csv')
location = pd.read_csv('location_features.csv')
print('food_final:', final.shape, ' location_features:', location.shape)

food_final: (126787, 53)  location_features: (126787, 3)


food_final: **126787 rows and 53 columns**,  location_features: **126787 rows and 3 columns**. Both files have one row per inspection.

In [ ]:
data = final.merge(location[['Inspection ID', 'Zip_Group']], on='Inspection ID', how='left', validate='one_to_one')
print(data.shape)
print('Inspections without a Zip_Group:', data['Zip_Group'].isnull().sum())

(126787, 54)
Inspections without a Zip_Group: 0


We add the `Zip_Group` column (the decision in `08_Location_Check.ipynb`) with the same merge as in `06_Merge.ipynb`. `validate='one_to_one'` checks that every inspection appears only once in both files. There are **0** inspections without a Zip_Group.

## 1. Choose the features

Only columns that are known **before** the inspection can be used (✅ in the README). The columns about the inspection itself (Results, Violations, Num_Violations, has_… topics) are known only after it, so using them would be **leakage**.

| Feature | Type | Why |
| --- | --- | --- |
| Prev_Fail, Past_Fail_Rate | number | the business's own history of failing |
| Prev_Critical, Prev_Serious, Prev_Num_Violations, Past_Critical_Rate, Past_Serious_Rate | number | violations at earlier visits |
| Days_Since_Last, Num_Past_Inspections, Days_Since_First_Seen | number | how often and since when the business is inspected |
| Is_Reinspection | 0 / 1 | re-inspections follow a failed inspection |
| Risk_Num | number | official risk category (1 High – 3 Low) |
| Facility_Group, Inspection_Group, Season, Zip_Group | category → dummies | type of place, type of visit, time of year, location |

**Left out:** `First_Inspection` (same information as Num_Past_Inspections = 0, see the Phase 1 report) and `Month` (Season already describes the time of year; Month as a number 1–12 would mean that December is "12 times" January).

In [ ]:
numeric_features = ['Prev_Fail', 'Past_Fail_Rate', 'Prev_Critical', 'Prev_Serious', 'Prev_Num_Violations',
                    'Past_Critical_Rate', 'Past_Serious_Rate', 'Days_Since_Last', 'Num_Past_Inspections',
                    'Days_Since_First_Seen', 'Is_Reinspection', 'Risk_Num']
category_features = ['Facility_Group', 'Inspection_Group', 'Season', 'Zip_Group']
keep_columns = ['Inspection ID', 'Inspection Date', 'Period', 'Placebo_Period', 'Fail']

data = data[keep_columns + numeric_features + category_features]
print(data.shape)

(126787, 21)


**126787 rows and 21 columns** (5 columns to split the data and measure the result, 12 numeric features and 4 categorical features).

In [ ]:
data.isnull().sum()[data.isnull().sum() > 0]

Series([], dtype: int64)

Missing values in the chosen columns. There are **no** missing values, because the history columns were already filled in `04_History.ipynb` and the Zip groups in `08_Location_Check.ipynb`.

In [ ]:
for col in category_features:
    print(col, ':', data[col].nunique(), 'groups')

Facility_Group : 9 groups
Inspection_Group : 4 groups
Season : 4 groups
Zip_Group : 58 groups


| Group | Entries |
| ----- | ------- |
| Facility_Group | Restaurant, Grocery Store, School, Children's Services Facility, Bakery, Daycare Above and Under 2 Years, Daycare (2 - 6 Years), Long Term Care, Other |
| Inspection_Group | Canvass, License, Complaint, Other |
| Season | Spring, Summer, Fall, Winter|
| Zip_Group | 60614, 60647, 60657, 60611, ... |

There are **9** facility, **4** inspection, **4** season and **58** Zip groups. Each of them becomes one dummy column per group, minus one (next step).

## 2. Dummy variables

In [ ]:
data = pd.get_dummies(data, columns=category_features, drop_first=True, dtype=int)
print(data.shape)
data.head()

(126787, 88)


,Inspection ID,Inspection Date,Period,Placebo_Period,Fail,Prev_Fail,Past_Fail_Rate,Prev_Critical,Prev_Serious,Prev_Num_Violations,...,Zip_Group_60653,Zip_Group_60654,Zip_Group_60655,Zip_Group_60656,Zip_Group_60657,Zip_Group_60659,Zip_Group_60660,Zip_Group_60661,Zip_Group_60666,Zip_Group_Other
0,2569129,2022-12-30,Post,After 2017,0,1.0,1.000000,1.0,1.0,5.0,...,0,0,0,0,0,0,0,0,0,0
1,2569114,2022-12-30,Post,After 2017,0,0.0,0.000000,0.0,1.0,3.0,...,0,0,0,0,0,0,0,0,0,0
2,2569125,2022-12-30,Post,After 2017,0,0.0,0.000000,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
3,2569122,2022-12-30,Post,After 2017,1,0.0,0.259259,0.0,0.0,3.0,...,0,0,0,0,1,0,0,0,0,0
4,2569126,2022-12-30,Post,After 2017,0,0.0,0.333333,0.0,1.0,6.0,...,0,0,0,0,0,1,0,0,0,0


`pd.get_dummies` turns every group into its own 0/1 column. With `drop_first=True` we drop one group of each column: if all the other dummies of a column are 0, the business is in the dropped group. Without this, the dummies of one column always add up to 1, so they are perfectly correlated (Chapter 12, "correlated features": c1 + c2 + c3 = 1). New shape: **126787 rows and 88 columns**

In [ ]:
dummy_columns = [c for c in data.columns if c not in keep_columns + numeric_features]
print('Number of dummy columns:', len(dummy_columns))
print(dummy_columns[:15])

Number of dummy columns: 71
["Facility_Group_Children's Services Facility", 'Facility_Group_Daycare (2 - 6 Years)', 'Facility_Group_Daycare Above and Under 2 Years', 'Facility_Group_Grocery Store', 'Facility_Group_Long Term Care', 'Facility_Group_Other', 'Facility_Group_Restaurant', 'Facility_Group_School', 'Inspection_Group_Complaint', 'Inspection_Group_License', 'Inspection_Group_Other', 'Season_Spring', 'Season_Summer', 'Season_Winter', 'Zip_Group_60602']


There are **71** dummy columns (8 facility, 3 inspection, 3 season and 57 Zip: each column has one group less than in the table above). Most of them are Zip groups. Together with the 5 columns to split the data and the 12 numeric features, this gives the 88 columns.

## 3. Train and test periods

In [ ]:
print(data['Period'].value_counts())
print()
print(data.groupby('Period')['Fail'].mean().round(3))

Period
Pre                64293
Post               47780
Late transition    11352
Grace (90 days)     3362
Name: count, dtype: int64

Period
Grace (90 days)    0.217
Late transition    0.209
Post               0.237
Pre                0.216
Name: Fail, dtype: float64


Number of inspections and fail rate per period. The fail rate is **21.6%** in Pre, **21.7%** in the Grace period, **20.9%** in the late transition and **23.7%** in Post (the same values as in H1a, `07_Tests`). So the fail rate is a little higher after the change, but in every period only about **1 of 5** inspections fails. The data is **imbalanced**: a model that always predicts "Pass" would already reach about 78% accuracy on Pre and 76% on Post without finding any failing restaurant (Chapter 12, "Is accuracy enough?"). This is why we will compare the models with **sensitivity, specificity and PPV**, not only accuracy.

We use the periods like this:

| Test | Train on | Test on |
| --- | --- | --- |
| H2 (main question) | Pre (80%) | Pre (the other 20%) **and** Post |
| H3 (placebo) | Pre before 1 Jan 2017 | Pre after 1 Jan 2017 |
| Reference (random split) | random 80% of Pre + Post together | the other 20% |

The **random split** mixes the years, so the model sees inspections from after the change during training. It usually looks best, so it shows how good the model would look if we ignored time; the real question is how much worse the Pre → Post result is. Inside the training data we also use **k-fold cross-validation** (Chapter 10) to choose the model settings (for example k in KNN).

Grace and Late transition are left out of the models: they are a mix of the old and the new system.

In [ ]:
pre = data[data['Period'] == 'Pre']
print(pre['Placebo_Period'].value_counts())
print()
print(pre.groupby('Placebo_Period')['Fail'].mean().round(3))

Placebo_Period
Before 2017    37592
After 2017     26701
Name: count, dtype: int64

Placebo_Period
After 2017     0.226
Before 2017    0.209
Name: Fail, dtype: float64


The two parts of the Pre period for the placebo test (H3): **37,592** inspections before 1 January 2017 with a fail rate of **20.9%**, and **26,701** after it with **22.6%**. So even **without** a change of the law, the fail rate went up by **1.7 percentage points**, almost as much as from Pre to Post (21.6% → 23.7%, +2.1 points). The fail rate changes a little over time anyway, so a lower model performance after 2018 does **not** prove by itself that the new food code is the reason. This is exactly why we need the placebo test: only if the drop at the real change (2018) is clearly **larger** than the drop at the placebo date (2017) can we say that the new code made the model worse.

In [ ]:
data.to_csv('model_data.csv', index=False)
print('Saved:', data.shape)

Saved: (126787, 88)


We save the model data. The scaling (needed for KNN) is done in the model notebook, **after** splitting, using only the training data, so no information from the test data leaks into the training.